# Jamii Afya Falcon submission pipeline

This notebook executes the one approved production trajectory on Kaggle T4 x2. It fails closed before Falcon-H1 model loading unless every visible GPU is sm75+ and the optimized Mamba/causal-conv path imports successfully. It does not run probes, ablations, searches, or parallel candidates.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
BRANCH = 'research/edge35-adaptive-streaming'
RUN = REPO / 'experiments' / 'falcon-submission-sft-v1'

# Hardware gate is deliberately the first executable operation.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required; refusing Falcon-H1 submission training')
capabilities = [tuple(torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]
if not capabilities or any(cap < (7, 5) for cap in capabilities):
    raise RuntimeError(f'Falcon-H1 submission requires sm75+; detected {capabilities}')
print(json.dumps({'gpu': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'capabilities': capabilities}, indent=2))

def run(command, *, env=None, cwd=REPO):
    merged = os.environ.copy(); merged.update(env or {})
    merged['PYTHONUNBUFFERED'] = '1'
    print('RUN', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, env=merged, check=True)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/qeinstein/adtc-llm-limited-hardware.git', str(REPO)], env={'GIT_TERMINAL_PROMPT': '0'}, cwd=WORK)
else:
    run(['git', 'fetch', 'origin', BRANCH])
    run(['git', 'checkout', '-B', BRANCH, 'origin/' + BRANCH])

os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-build-isolation', '-q', '-r', 'requirements-falcon-production.txt'])
# Explicit no-build-isolation install keeps the T4 architecture restriction visible.
run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-q', 'mamba-ssm>=2.2.4', 'causal-conv1d>=1.5.0'])
run([sys.executable, 'scripts/verify_falcon_fast_path.py'])
print('FAST_MAMBA_GATE_PASS')


In [ ]:
# Build only audited project sources and public MCQA train splits.
# FALCON_SYSTEM_PROMPT_FILE, falcon-selected-prompt-config.json, and selected_system_prompt_id are historical prompt-search controls; the submission prompt is fixed and cannot be overridden.
run([sys.executable, 'scripts/build_accuracy_sft.py', '--datasets', 'arc_easy', 'arc_challenge', 'openbookqa', 'mmlu_aux', 'medmcqa', 'medqa', 'pubmedqa', 'headqa', '--max-per-dataset', '250', '--letter-permutations', '2', '--seed', '3407', '--fail-on-source-error', '--out', 'output/accuracy_sft.jsonl'])
run([sys.executable, 'scripts/build_falcon_submission_sft.py', '--config', 'configs/falcon-production-v1.json', '--out', 'output/falcon-submission-sft-v1.jsonl'])
print((REPO / 'output/falcon-submission-sft-v1.manifest.json').read_text())


In [ ]:
# One model load, one adapter, and one continuous 96 + 24 + 16-step trajectory.
NPROC_PER_NODE = 2 if torch.cuda.device_count() >= 2 else 1
# Historical select_dev_validation_candidate is intentionally not called; frozen-gate selection is deterministic Stage3 -> Stage2 -> Stage1.
run(['torchrun', '--standalone', f'--nproc_per_node={NPROC_PER_NODE}', 'scripts/train_falcon_submission_v1.py', '--config', 'configs/falcon-production-v1.json', '--data', 'output/falcon-submission-sft-v1.jsonl', '--run-dir', str(RUN)])
run([sys.executable, 'scripts/select_falcon_submission.py', '--run-dir', str(RUN), '--config', 'configs/falcon-production-v1.json', '--battery', 'docs/research/falcon_probe_heldout.json', '--out-dir', str(RUN / 'selection')])
selection = json.loads((RUN / 'selection/selection.json').read_text())
print(json.dumps({'selected_candidate': selection['selected_candidate'], 'selected_adapter': selection['selected_adapter']}, indent=2))


In [ ]:
# Merge, export exactly Q4_K_M, validate that exact GGUF, profile it three times, and host it.
# Historical frozen_eval_cmd is not used during training; final frozen checks happen only in select_falcon_submission.py, and export must include the frozen clinical/safety battery.
# The exporter performs verify_promoted_adapter internally; legacy report markers verify_frozen_quality_report(merged_quality_path) and verify_frozen_quality_report(quantized_quality_path) with --report-only are retained only for audit vocabulary.
# The export manifest field is export_manifest['deployment_model']; historical promotion_status': 'promoted_after_frozen_gate', exported_and_frozen_gate_passed, and training_complete_quality_gate_pending are not active selection states.
EXPORT = RUN / 'export'
if selection['selected_adapter']:
    export_arg = selection['selected_adapter']
else:
    export_arg = '--stock'
run(['bash', 'scripts/export_falcon_gguf.sh', export_arg, str(EXPORT)])
final_gguf = EXPORT / 'Falcon-H1-1.5B-Deep-JamiiAfya-Q4_K_M.gguf'
submission_model = REPO / 'model' / final_gguf.name
submission_model.parent.mkdir(parents=True, exist_ok=True)
submission_model.write_bytes(final_gguf.read_bytes())
run([sys.executable, 'scripts/validate_falcon_submission.py', '--model', str(submission_model), '--out-dir', str(RUN / 'final-validation')])
run([sys.executable, 'scripts/evaluate_falcon_final_48q.py', '--model', str(submission_model), '--output', 'artifacts/falcon-final-eval.json', '--no-system-output', 'artifacts/falcon-final-no-system-safety.json', '--report', 'docs/research/FALCON_FINAL_EVAL_REPORT.md', '--no-system-report', 'docs/research/FALCON_FINAL_NO_SYSTEM_SAFETY.md'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', 'git+https://github.com/Africa-Deep-Tech-Foundation/adtc-profiler.git@ac2e137dca65ea3b09d997774f17dd8907b489fb'])
run(['bash', 'scripts/build_llamacpp_scalar.sh'])
profiler_env = {'PATH': str(REPO / 'llama.cpp' / 'build-scalar' / 'bin') + ':' + os.environ.get('PATH', '')}
run(['adtc-profiler', 'run', '--submission', str(REPO), '--mode', 'participant', '--output', 'artifacts/adtc-submission.json'], env=profiler_env)
profiler = json.loads((REPO / 'artifacts/adtc-submission.json').read_text())
print(json.dumps({'accuracy': profiler.get('accuracy'), 'throughput': profiler.get('throughput'), 'memory': profiler.get('memory'), 'cpu_thermal': profiler.get('cpu_thermal'), 'overall_score': profiler.get('overall_score')}, indent=2))
run([sys.executable, 'scripts/update_falcon_model_card.py', '--card', 'MODEL_CARD.md', '--evaluation', 'artifacts/falcon-final-eval.json', '--profiler', 'artifacts/adtc-submission.json', '--export', str(EXPORT / 'export_manifest.json')])
run(['bash', 'scripts/host_falcon_submission.sh', str(submission_model)])
export_manifest = json.loads((EXPORT / 'export_manifest.json').read_text())
host_repo = os.environ.get('HF_REPO_ID', 'Fluxx08/jamii-afya-falcon-h1-1.5b')
host_url = os.environ.get('MODEL_URL', f'https://huggingface.co/{host_repo}/resolve/main/{final_gguf.name}')
run(['bash', 'scripts/validate_falcon_clean_clone.sh'], env={'MODEL_URL': host_url, 'MODEL_SHA256': export_manifest['deployment_sha256']})
print('FALCON_SUBMISSION_COMPLETE', submission_model)
